# Generate Fake SportLogiq Data → UC Volume

Skip notebooks **00** and **01**, run this notebook, then resume at **02**.

This notebook fakes the response shapes from the SportLogiq Hockey API so you
can exercise bronze→gold→dashboard→Genie without API credentials. Files land at
the same Volume paths the real ingest would write to, so notebooks 02-07 work
unchanged.

> Numbers are synthetic — random values seeded to `42` for reproducibility.
> The shapes match what `03_silver_transformations.ipynb` extracts; if the real
> SportLogiq payload diverges, that'll surface as `_rescued` rows on first real
> ingest.

In [1]:
import os, json, random
from datetime import date, timedelta
from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(os.path.abspath('.')), '.env'))   # tests/ -> demo root .env
load_dotenv()                                                              # also try cwd

# Dual-mode Spark: works locally via Databricks Connect AND in workspace.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

random.seed(42)

In [2]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "sportlogiq_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Volume:  {VOLUME_PATH}")

Catalog: alexander_booth
Volume:  /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data


## 1. Create schemas + Volume (same as notebook 00, minus the SportLogiq probe)

In [3]:
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}")
    print(f"  schema ready: {UC_CATALOG}.{schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")
print(f"  volume ready: {VOLUME_PATH}")

  schema ready: alexander_booth.sportlogiq_nhl_bronze


  schema ready: alexander_booth.sportlogiq_nhl_silver


  schema ready: alexander_booth.sportlogiq_nhl_gold


  volume ready: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data


In [4]:
def upload_json(path, payload):
    body = json.dumps(payload).encode("utf-8")
    w.files.upload(path, body, overwrite=True)
    return len(body)


print("Uploader ready.")

Uploader ready.


## 2. Reference data — teams, venues, players, metric topics, competition, team records

In [5]:
SEASON         = "20232024"
COMPETITION_ID = "1"

# 8 teams (full league is 32; this is enough for variety)
TEAMS = [
    # (id, shorthand, location, name, division, conference)
    (1,  "BOS", "Boston",     "Bruins",         "Atlantic",     "Eastern"),
    (2,  "BUF", "Buffalo",    "Sabres",         "Atlantic",     "Eastern"),
    (3,  "TOR", "Toronto",    "Maple Leafs",    "Atlantic",     "Eastern"),
    (4,  "TBL", "Tampa Bay",  "Lightning",      "Atlantic",     "Eastern"),
    (5,  "NYR", "New York",   "Rangers",        "Metropolitan", "Eastern"),
    (6,  "CHI", "Chicago",    "Blackhawks",     "Central",      "Western"),
    (7,  "COL", "Colorado",   "Avalanche",      "Central",      "Western"),
    (8,  "VGK", "Vegas",      "Golden Knights", "Pacific",      "Western"),
]

# 8 venues, one per team
VENUES = [
    (1, "TD Garden",        "Boston",     "MA", "USA", "America/New_York",     17850),
    (2, "KeyBank Center",   "Buffalo",    "NY", "USA", "America/New_York",     19070),
    (3, "Scotiabank Arena", "Toronto",    "ON", "CAN", "America/Toronto",      18819),
    (4, "Amalie Arena",     "Tampa Bay",  "FL", "USA", "America/New_York",     19092),
    (5, "Madison Square Garden","New York","NY","USA","America/New_York",      18006),
    (6, "United Center",    "Chicago",    "IL", "USA", "America/Chicago",      19717),
    (7, "Ball Arena",       "Denver",     "CO", "USA", "America/Denver",       18007),
    (8, "T-Mobile Arena",   "Las Vegas",  "NV", "USA", "America/Los_Angeles",  17500),
]

# 50 players spread across the 8 teams, varied positions
POSITIONS  = ["F"] * 4 + ["D"] * 2 + ["G"]            # weighted: 4F / 2D / 1G per slot
ROLES      = {"F": "forward", "D": "defenseman", "G": "goalie"}
FIRST = ["Connor","Auston","Nathan","David","Sidney","Patrick","Erik","Mikko",
        "Jack","Quinn","Cale","Kirill","Brad","Jason","Mark","Logan","Ryan",
        "Tom","Jordan","Adam","Sam","Tyler","Mason","Dylan","Liam"]
LAST  = ["McDavid","Matthews","MacKinnon","Pastrnak","Crosby","Kane","Karlsson",
         "Rantanen","Hughes","Eichel","Makar","Kaprizov","Marchand","Robertson",
         "Stone","Couture","Stamkos","Wilson","Kyrou","Fox","Reinhart","Seguin"]

players = []
pid = 100
for team_id, sh, loc, name, div, conf in TEAMS:
    for slot_index in range(6):                              # 6 players per team
        pos = POSITIONS[slot_index] if slot_index < len(POSITIONS) else "F"
        players.append({
            "id":              str(pid),
            "first_name":      random.choice(FIRST),
            "last_name":       random.choice(LAST),
            "current_team_id": str(team_id),
            "position":        pos,
            "role":            ROLES[pos],
            "status":          "active",
        })
        pid += 1
# top-up to 50
while len(players) < 50:
    players.append({
        "id":              str(pid),
        "first_name":      random.choice(FIRST),
        "last_name":       random.choice(LAST),
        "current_team_id": str(random.choice([t[0] for t in TEAMS])),
        "position":        "F",
        "role":            "forward",
        "status":          "active",
    })
    pid += 1

print(f"prepared: {len(TEAMS)} teams, {len(VENUES)} venues, {len(players)} players")

prepared: 8 teams, 8 venues, 50 players


In [6]:
# 1 — Competition detail (silver_competitions reads data:id::string from this file)
upload_json(
    f"{VOLUME_PATH}/reference/competition_{COMPETITION_ID}.json",
    {
        "id": COMPETITION_ID, "name": "NHL",
        "seasons": [
            {"name": SEASON, "stages": [
                {"name": "regular", "start_date": "2023-10-10", "end_date": "2024-04-15"}
            ]}
        ],
    },
)

# 2 — Teams list
upload_json(
    f"{VOLUME_PATH}/reference/teams.json",
    {
        "competition_id": int(COMPETITION_ID),
        "season": SEASON, "stage": "regular",
        "teams": [
            {"id": str(tid), "name": f"{loc} {nm}", "location": loc,
             "shorthand": sh, "division": div, "conference": conf,
             "logo_src": f"https://example.com/logos/{sh.lower()}.png"}
            for (tid, sh, loc, nm, div, conf) in TEAMS
        ],
    },
)

# 3 — Team records
upload_json(
    f"{VOLUME_PATH}/reference/team_records.json",
    {
        "teams": [
            {
                "id": str(tid),
                "record": {
                    "wins":            random.randint(20, 40),
                    "losses":          random.randint(15, 30),
                    "overtime_losses": random.randint(2, 8),
                    "ties":            0,
                },
            }
            for (tid, *_) in TEAMS
        ]
    },
)

# 4 — Venues
upload_json(
    f"{VOLUME_PATH}/reference/venues.json",
    {
        "venues": [
            {"id": str(vid), "name": nm, "city": city, "state": st,
             "country": cn, "timezone": tz, "capacity": cap}
            for (vid, nm, city, st, cn, tz, cap) in VENUES
        ]
    },
)

# 5 — Players (one season-pull file)
upload_json(
    f"{VOLUME_PATH}/reference/players_{SEASON}.json",
    {"season": SEASON, "players": players},
)

# 6 — Metric topics, one file per scope
TOPIC_IDS = {"scoring": "1", "defense": "2"}
for scope in ["player", "team", "goalie", "opposingteam"]:
    upload_json(
        f"{VOLUME_PATH}/reference/metric_topics_{scope}.json",
        {
            "scope":  scope,
            "topics": [{"id": tid, "name": name} for name, tid in TOPIC_IDS.items()],
        },
    )

print("Reference data uploaded.")

Reference data uploaded.


## 3. Per-game data (5 fake games)

In [7]:
# (game_id, date, home_team_id, away_team_id, venue_id, home_score, away_score, winning_team_id)
GAMES = [
    (1001, "2023-10-15", 2, 1, 2, 4, 2, 2),
    (1002, "2023-10-17", 1, 3, 1, 3, 1, 1),
    (1003, "2023-10-20", 5, 4, 5, 2, 3, 4),
    (1004, "2023-10-22", 8, 7, 8, 5, 4, 8),
    (1005, "2023-10-25", 7, 6, 7, 1, 0, 7),
]

TEAMS_BY_ID  = {t[0]: t for t in TEAMS}
PLAYERS_BY_TEAM = {tid: [p for p in players if int(p["current_team_id"]) == tid] for tid, *_ in TEAMS}
EVENT_NAMES  = ["pass", "shot", "faceoff", "hit", "block", "giveaway", "takeaway", "save"]
EVENT_TYPES  = ["regular", "regular", "regular", "regular", "powerplay"]
SHOT_OUTCOMES = ["save", "save", "save", "missed", "blocked", "goal"]   # ~17% goal rate
ZONES        = ["O", "D", "N"]


def fake_event(game_id, period, sequence_id, team_id, opp_team_id, home_team_id):
    """Build one synthetic compiled-event payload (~30 fields)."""
    pool = PLAYERS_BY_TEAM[team_id]
    p = random.choice(pool)
    is_home = team_id == home_team_id
    name = random.choices(EVENT_NAMES, weights=[3,3,1,1,1,1,1,1])[0]
    is_shot = name in ("shot", "save")
    outcome = random.choice(SHOT_OUTCOMES) if is_shot else random.choice(["complete","incomplete"])
    # x in [-100,100], y in [-50,50]; bias shots into the offensive zone
    if is_shot:
        x = random.randint(40, 90)  if is_home else random.randint(-90, -40)
        zone = "O"
    else:
        x = random.randint(-90, 90)
        zone = random.choice(ZONES)
    y = random.randint(-40, 40)
    period_time = random.randint(0, 1199)
    return {
        "event_id":         f"E{game_id}{period}{sequence_id:04d}",
        "game_reference_id": str(game_id),
        "x_coord": x, "y_coord": y, "zone": zone,
        "timecode": f"{period_time//60:02d}:{period_time%60:02d}",
        "frame":  period_time * 30,
        "period": period,
        "period_time": period_time,
        "game_time":   (period - 1) * 1200 + period_time,
        "minutes": period_time // 60,
        "seconds": period_time % 60,
        "team":              TEAMS_BY_ID[team_id][1],   # shorthand
        "player_id":         p["id"],
        "player_position":   p["position"],
        "player_first_name": p["first_name"],
        "player_last_name":  p["last_name"],
        "player_jersey":     str(random.randint(1, 99)),
        "event_sequence_id": str(sequence_id),
        "shorthand":  name[:4].upper(),
        "name":       name,
        "type":       random.choice(EVENT_TYPES),
        "outcome":    outcome,
        "flags":      [],
        "players_involved_in_play":          json.dumps([p["id"]]),
        "opposing_team_players_involved_in_play": "[]",
        "current_possession":             sequence_id // 5,
        "team_in_possession":             TEAMS_BY_ID[team_id][1],
        "current_play_in_possession":     sequence_id % 5,
        "current_off_play_in_possession": sequence_id % 3,
        "current_def_play_in_possession": sequence_id % 2,
        "related_event_id":         None,
        "related_event_player_id":  None,
        "related_event_player_first_name": None,
        "related_event_player_last_name":  None,
        "related_event_player_number":     None,
        "attributes": {},
    }


def fake_shift(game_id, period, sequence_id, team_id):
    pool = PLAYERS_BY_TEAM[team_id]
    p = random.choice(pool)
    period_time = random.randint(0, 1199)
    return {
        "event_id":          f"S{game_id}{period}{sequence_id:04d}",
        "event_sequence_id": str(sequence_id),
        "period":      period,
        "period_time": period_time,
        "team":      TEAMS_BY_ID[team_id][1],
        "player_id": p["id"],
        "type":     random.choice(["on_ice", "off_ice"]),
        "outcome":  "complete",
        "x_coord":  random.randint(-90, 90),
        "y_coord":  random.randint(-40, 40),
        "zone":     random.choice(ZONES),
    }


def fake_toi_entry(game_id, period, player, team_shorthand):
    return {
        "player_id":   player["id"],
        "team":        team_shorthand,
        "period":      period,
        "strength":    random.choice(["5v5", "5v5", "5v5", "5v4", "4v5"]),
        "toi_seconds": random.randint(60, 700),
    }


def fake_metric_grouping(player, team_id, period, position, topic_id):
    return {
        "breakdown": [
            {"key": "player",   "value": player["id"]},
            {"key": "team",     "value": str(team_id)},
            {"key": "period",   "value": str(period)},
            {"key": "position", "value": position},
        ],
        "metrics": [
            {"key": "corsi_for",      "value": random.uniform(0, 25)},
            {"key": "fenwick_for",    "value": random.uniform(0, 20)},
            {"key": "expected_goals", "value": random.uniform(0, 1.5)},
        ],
        "unitsplayed": [{"name": "minutes", "value": random.uniform(0, 20)}],
    }


print("Builders ready.")

Builders ready.


In [8]:
total_files = 0

for (game_id, dt, home_id, away_id, venue_id, hs, as_, win_id) in GAMES:
    base = f"{VOLUME_PATH}/games/{game_id}"
    home, away = TEAMS_BY_ID[home_id], TEAMS_BY_ID[away_id]

    # ---- game.json (single object — one bronze row)
    upload_json(f"{base}/game.json", {
        "id":         str(game_id),
        "season":     SEASON,
        "stage":      "regular",
        "date":       dt,
        "scheduled_time":  f"{dt}T19:00:00Z",
        "venue_id":   str(venue_id),
        "home_team":  {"id": str(home_id), "name": f"{home[2]} {home[3]}", "location": home[2],
                       "shorthand": home[1], "division": home[4], "conference": home[5],
                       "logo_src": f"https://example.com/logos/{home[1].lower()}.png"},
        "away_team":  {"id": str(away_id), "name": f"{away[2]} {away[3]}", "location": away[2],
                       "shorthand": away[1], "division": away[4], "conference": away[5],
                       "logo_src": f"https://example.com/logos/{away[1].lower()}.png"},
        "home_score":      [hs],
        "away_score":      [as_],
        "winning_team_id": str(win_id),
    })
    total_files += 1

    # ---- roster.json (single object with crew array)
    crew = []
    for tid in (home_id, away_id):
        for p in PLAYERS_BY_TEAM[tid][:5]:
            crew.append({"id": p["id"],
                         "first_name": p["first_name"],
                         "last_name":  p["last_name"],
                         "position":   p["position"],
                         "task":       "player",
                         "role":       p["role"]})
    crew.extend([
        {"id": "9001", "first_name": "Wes",  "last_name": "McCauley",  "position": "official", "task": "referee", "role": None},
        {"id": "9002", "first_name": "Chris","last_name": "Rooney",    "position": "official", "task": "linesman", "role": None},
    ])
    upload_json(f"{base}/roster.json", {"crew": crew})
    total_files += 1

    # ---- compiled_events.json (TOP-LEVEL ARRAY — Auto Loader fans out to N rows)
    compiled = []
    for period in (1, 2, 3):
        for seq in range(10):
            team = random.choice([home_id, away_id])
            opp  = away_id if team == home_id else home_id
            compiled.append(fake_event(game_id, period, seq, team, opp, home_id))
    upload_json(f"{base}/compiled_events.json", compiled)
    total_files += 1

    # ---- full_events.json (similar shape; smaller volume)
    full_events = compiled[:10]    # subset is fine — same schema
    upload_json(f"{base}/full_events.json", full_events)
    total_files += 1

    # ---- shift_events.json (top-level array)
    shifts = []
    for period in (1, 2, 3):
        for seq in range(5):
            team = random.choice([home_id, away_id])
            shifts.append(fake_shift(game_id, period, seq, team))
    upload_json(f"{base}/shift_events.json", shifts)
    total_files += 1

    # ---- player_toi.json (top-level array; silver direct-extracts each row)
    toi = []
    for period in (1, 2, 3):
        for tid in (home_id, away_id):
            for p in PLAYERS_BY_TEAM[tid][:4]:
                toi.append(fake_toi_entry(game_id, period, p, TEAMS_BY_ID[tid][1]))
    upload_json(f"{base}/player_toi.json", toi)
    total_files += 1

    # ---- per-topic player metrics (one file per topic id)
    for topic_name, topic_id in TOPIC_IDS.items():
        groupings = []
        for tid in (home_id, away_id):
            for p in PLAYERS_BY_TEAM[tid][:4]:
                for period in (1, 2, 3):
                    groupings.append(fake_metric_grouping(p, tid, period, p["position"], topic_id))
        upload_json(f"{base}/metrics_player_{topic_id}.json", {
            "id":              str(game_id),
            "season":          SEASON,
            "stage":           "regular",
            "competition_id":  COMPETITION_ID,
            "topic_id":        topic_id,
            "groupings":       groupings,
        })
        total_files += 1

print(f"Per-game files written: {total_files} (across {len(GAMES)} games)")

Per-game files written: 40 (across 5 games)


## 4. Season-aggregate metrics (4 scopes × 2 topics each)

In [9]:
def fake_season_grouping(scope, season, topic_id):
    if scope in ("player", "goalie"):
        ent_pool = [(p["id"], p["current_team_id"], p["position"]) for p in players]
    else:                                          # team / opposingteam
        ent_pool = [(None, str(t[0]), None) for t in TEAMS]
    g = []
    for ent_id, team_id, pos in ent_pool:
        breakdown = []
        if ent_id is not None:
            breakdown.append({"key": "player", "value": ent_id})
        breakdown.append({"key": "team", "value": team_id})
        if pos is not None:
            breakdown.append({"key": "position", "value": pos})
        g.append({
            "breakdown":   breakdown,
            "metrics": [
                {"key": "corsi_for",      "value": random.uniform(50, 600)},
                {"key": "fenwick_for",    "value": random.uniform(40, 500)},
                {"key": "expected_goals", "value": random.uniform(0, 30)},
            ],
            "unitsplayed": [{"name": "minutes", "value": random.uniform(60, 1200)}],
        })
    return g


for scope in ["player", "team", "goalie", "opposingteam"]:
    for topic_name, topic_id in TOPIC_IDS.items():
        upload_json(
            f"{VOLUME_PATH}/season_metrics/{scope}/{topic_id}.json",
            {
                "season":         SEASON,
                "stage":          "regular",
                "competition_id": COMPETITION_ID,
                "topic_id":       topic_id,
                "groupings":      fake_season_grouping(scope, SEASON, topic_id),
            },
        )
print("Season-metric files written: 8 (4 scopes × 2 topics).")

Season-metric files written: 8 (4 scopes × 2 topics).


## 5. Verify

In [10]:
def count_under(prefix):
    try:
        return sum(1 for _ in w.files.list_directory_contents(prefix))
    except Exception:
        return 0


print("=" * 60)
print("FAKE DATA SUMMARY")
print("=" * 60)
for sub in ["reference", "games", "season_metrics"]:
    n = count_under(f"{VOLUME_PATH}/{sub}")
    print(f"  {sub:<20s}  {n} entries")

# Drill into one game directory to confirm shape
sample_game = GAMES[0][0]
print(f"\n  games/{sample_game} contents:")
try:
    for f in w.files.list_directory_contents(f"{VOLUME_PATH}/games/{sample_game}"):
        size_kb = (f.file_size or 0) / 1024
        print(f"    {f.name:<35s}  {size_kb:6.1f} KB")
except Exception as e:
    print(f"    (could not list: {e})")

print(f"\nVolume root: {VOLUME_PATH}")
print("Next step: run 02_bronze_autoloader.ipynb.")

FAKE DATA SUMMARY


  reference             9 entries


  games                 5 entries


  season_metrics        4 entries

  games/1001 contents:


    compiled_events.json                   27.1 KB
    full_events.json                        9.0 KB
    game.json                               0.6 KB
    metrics_player_1.json                   9.2 KB
    metrics_player_2.json                   9.2 KB
    player_toi.json                         2.1 KB
    roster.json                             1.4 KB
    shift_events.json                       3.0 KB

Volume root: /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data
Next step: run 02_bronze_autoloader.ipynb.


## 6. Optional reset

Drops the demo schemas and the Volume so re-running the test starts clean.
Uncomment to use.

In [11]:
# for schema in [GOLD_SCHEMA, SILVER_SCHEMA, BRONZE_SCHEMA]:
#     spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{schema} CASCADE")
#     print(f"  dropped {UC_CATALOG}.{schema}")
# print("Demo schemas dropped. Re-run cells 1-5 to regenerate.")